# 01 — Data preparation and validation
 **Question:** Are the hourly and daily Capital Bikeshare data internally consistent and ready for visualization?

 **Source:** Fanaee-T & Gama (2013), *Bike Sharing Dataset*, UCI Machine Learning Repository; Capital Bikeshare, Washington D.C. (2011–2012).

 **Method:** The raw datasets are not joined or appended: `day.csv` is the daily aggregation of the events described by `hour.csv`. We preserve both raw files unchanged. An absent hourly record is **not** automatically interpreted as zero rentals.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

hour = pd.read_csv('../data/raw/hour.csv')
day = pd.read_csv('../data/raw/day.csv')

for name, df in [('hour', hour), ('day', day)]:
    print(f'{name}: {len(df):,} rows, {len(df.columns)} columns')
    print(' Missing values:', int(df.isna().sum().sum()),
          '| Duplicate full rows:', int(df.duplicated().sum()))
    print(' Unique dates:', df['dteday'].nunique())

print('Duplicate (date, hour) keys:', int(hour.duplicated(['dteday', 'hr']).sum()))
print('Duplicate daily dates:', int(day.duplicated('dteday').sum()))
display(hour.head(3))

In [ ]:
# Create meaningful date/time fields; labels are only helpers for visualizations.
for df in [hour, day]:
    df['dteday'] = pd.to_datetime(df['dteday'])
    df['year'] = df['dteday'].dt.year
    df['month_name'] = df['dteday'].dt.month_name()
    df['day_name'] = df['dteday'].dt.day_name()

hour['datetime'] = hour['dteday'] + pd.to_timedelta(hour['hr'], unit='h')

weather_labels = {
    1: 'Clear / Partly Cloudy',
    2: 'Mist / Cloudy',
    3: 'Light Rain / Snow',
    4: 'Severe Weather'
}

for df in [hour, day]:
    df['weather_label'] = df['weathersit'].map(weather_labels)
    # Conversions documented in the source Readme.txt.
    df['temperature_c'] = df['temp'] * 41
    df['feels_like_c'] = df['atemp'] * 50
    df['humidity_pct'] = df['hum'] * 100

display(hour[['datetime','temperature_c','humidity_pct','weather_label','cnt']].head())

In [ ]:
# Dataset consistency checks
assert (hour['casual'] + hour['registered'] == hour['cnt']).all()
assert (day['casual'] + day['registered'] == day['cnt']).all()

daily_from_hour = hour.groupby('dteday')['cnt'].sum().sort_index()
daily_from_day = day.set_index('dteday')['cnt'].sort_index()
assert daily_from_hour.equals(daily_from_day)

# An absent (date, hour) record is not visible as NaN in the original file.
hourly_grid = hour.pivot_table(index='dteday', columns='hr', values='cnt', aggfunc='sum')
hourly_grid = hourly_grid.reindex(columns=range(24))
missing_hours = int(hourly_grid.isna().sum().sum())
incomplete_days = int(hourly_grid.isna().any(axis=1).sum())

print('Hourly missing cells in date x 24-hour grid:', missing_hours)
print('Days with at least one absent hourly record:', incomplete_days)
print('All rental totals consistent: True')
print('Weather code frequencies:')
display(hour['weather_label'].value_counts().to_frame('Observed hours'))

In [ ]:
from pathlib import Path

Path('../data/processed').mkdir(parents=True, exist_ok=True)
hour.to_csv('../data/processed/bike_hour_clean.csv', index=False)
day.to_csv('../data/processed/bike_day_clean.csv', index=False)
print('Saved bike_hour_clean.csv and bike_day_clean.csv.')

### Data quality decisions
 - No missing cell values or duplicate records were identified in the original files.
 - Some hourly **time slots are absent**: retain this distinction from recorded zero-demand hours.
 - Values in `temp`, `atemp` and `hum` are originally normalized; derived columns express understandable units.
 - We preserve `cnt = casual + registered` and exact daily aggregation.
 - **Limit:** rows are aggregate counts, not individual trips. We cannot know the actual purpose or destination of a rental.